<a href="https://colab.research.google.com/github/Nick97382000/Lettura_bilanci/blob/main/Lettura%20bilanci.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Lettura bilanci

In [16]:
!git clone https://github.com/Nick97382000/Lettura_bilanci
!pip install pdfplumber

fatal: destination path 'Lettura_bilanci' already exists and is not an empty directory.


In [17]:
import PyPDF2
import pdfplumber
import pandas as pd

pdf_path_2= "Lettura_bilanci/deposito bilanci/Kirey - 2024.pdf"

with open(pdf_path_2, "rb") as file:
    reader = PyPDF2.PdfReader(file)  # Read the PDF
    text = ""
    for page in reader.pages:
        text += page.extract_text() + ""

print(text[:100])

with pdfplumber.open(pdf_path_2) as pdf:
    page = pdf.pages[2]
    tables = page.extract_tables()

    if tables:
        t = tables[0]
        print(t.shape)
        if t:
            print("KO")
        t_clean = [row for row in t if any(cell is not None for cell in row)]
        max_cols = max(len(row) for row in t_clean)
        t_uniform = [row + [None] * (max_cols - len(row)) for row in t_clean]

        df= pd.DataFrame(t_uniform)
        print(df.shape)
        print(df)




In [18]:
import pdfplumber
import pandas as pd

def trova_tabella_bilancio(pdf_path, keywords = ["immobilizzazioni immateriali",
                                                 "crediti",
                                                 "immobilizzazioni materiali",
                                                 "immobilizzazioni finanziarie"]):
    with pdfplumber.open(pdf_path) as pdf:
        pagina_start = None
        for i, page in enumerate(pdf.pages):
            tables = page.extract_tables()
            for ii, t in enumerate(tables):
                testo = " ".join(
                    str(cell).lower()
                    for row in t
                    for cell in row
                    if cell
                )
                if all(kw in testo for kw in keywords):
                    pagina_start = i
                    break;
            if pagina_start is not None:
                break

        if pagina_start is None:
            return None

        tutte_le_righe = []
        intestazione = None

        for i in range (pagina_start, len(pdf.pages)):
                tables = pdf.pages[i].extract_tables()

                if not tables:
                    break

                for t in tables:
                    righe = [r for r in t if any(c for c in r)]
                    tutte_le_righe.extend(righe)

        if tutte_le_righe:
            n_colonne = max(len(r) for r in tutte_le_righe)
            tutte_le_righe = [r + [None]*(n_colonne - len(r)) for r in tutte_le_righe]

            df = pd.DataFrame(tutte_le_righe)
            return df
    return None


pdf_path_2= "Lettura_bilanci/deposito bilanci/Conserve Italia - Civilistico e Consolidato - 2024.pdf"
tabella = trova_tabella_bilancio(pdf_path_2)

print(tabella)

                                                     0  \
0                          STATO PATRIMONIALE - ATTIVO   
1                                                   A)   
2                                                        
3                                                        
4                                                   B)   
..                                                 ...   
299                                                      
300                                                      
301                                                      
302                                                      
303  Stato patrimoniale\nConto economico\nRendicont...   

                                                     1  \
0                                                 None   
1     CREDITI VERSO SOCI PER VERSAMENTI ANCORA DOVUTI:   
2    Quote capitale sociale sottoscritte e non versate   
3    TOTALE CREDITI V/SOCI PER VERSAMENTI ANCORA DO...   
4            

In [19]:
def summary_tabella(tabella):
    labels = [
        "Totale immobilizzazioni immateriali",
        "Totale immobilizzazioni materiali",
        "Totale immobilizzazioni finanziarie",
        "Totale patrimonio netto",
        "Totale disponibilità liquide a inizio esercizio",
        "Totale disponibilità liquide a inizio esercizio",
        "Totale debiti verso banche",
        "Differenza tra valore e costi della produzione",
        "Totale interessi e altri oneri finanziar",
        "Totale ammortamenti e svalutazioni",
        "Flusso finanziario dell'attività operativa",
        "Totale flusso finanziario dell'attività di investimento",
        "Flusso finanziario dell'attività di finanziamento"
    ]

    righe = {}

    if tabella.shape[1] == 3:
        col_label = 0
        cols_valori = [1, 2]
        date = tabella.iloc[0, 1:3].tolist()

    elif tabella.shape[1] == 5:
        col_label = 1
        cols_valori = [2, 3]
        date = tabella.iloc[0, 2:4].tolist()

    else:
        return pd.DataFrame()

    for label in labels:
        mask = tabella.iloc[:, col_label].astype(str).str.contains(
            label,
            case=False,
            na=False,
            regex=False
        )

        if mask.any():
            riga = tabella.loc[mask, cols_valori].iloc[0].tolist()
            righe[label] = riga

    return pd.DataFrame.from_dict(righe, orient="index", columns=date)


In [28]:
import pandas as pd
import numpy as np

def convert_bilancio_italiano(df):
    """Converte bilancio italiano → numeri"""

    # 1. Sostituisci parentesi TONDO (solo per valori negativi)
    df = df.replace({r'\(': '-', r'\)': ''}, regex=True)

    # 2. FUNZIONE per formato italiano: punti→niente, virgola→punto
    def to_numeric_it(x):
        if pd.isna(x) or x == '':
            return np.nan
        x = str(x).strip()
        # Rimuovi punti (migliaia), sostituisci virgola con punto
        x = x.replace('.', '').replace(',', '.')
        return pd.to_numeric(x, errors='coerce')

    # 3. Applica a tutto il DF
    return df.applymap(to_numeric_it)

# USO


In [30]:
import glob
import os
pdf_files = glob.glob("Lettura_bilanci/deposito bilanci/*.pdf")

risultati = {}
summary = {}

for pdf_path in pdf_files:
    nome = os.path.basename(pdf_path).replace(".pdf", "")
    tabella = trova_tabella_bilancio(pdf_path)
    if tabella is not None:
      summary[nome]=convert_bilancio_italiano(summary_tabella(tabella))
print(summary)

/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)


{'Conserve Italia - Civilistico e Consolidato - 2024':                                                     30 giugno 2025  \
Totale immobilizzazioni immateriali                     44577002.0   
Totale immobilizzazioni materiali                      342717358.0   
Totale immobilizzazioni finanziarie                    103068871.0   
Totale patrimonio netto                                305929558.0   
Totale disponibilità liquide a inizio esercizio        106847278.0   
Totale debiti verso banche                             191590518.0   
Differenza tra valore e costi della produzione          22431477.0   
Totale ammortamenti e svalutazioni                     -31210733.0   
Flusso finanziario dell'attività operativa              50640501.0   
Totale flusso finanziario dell'attività di inve...     -60736793.0   
Flusso finanziario dell'attività di finanziamento              NaN   

                                                    30 giugno 2024  
Totale immobilizzazioni immateriali

/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)


In [29]:
print(convert_bilancio_italiano(summary["Conserve Italia - Civilistico e Consolidato - 2024"]))

                                                    30 giugno 2025  \
Totale immobilizzazioni immateriali                     44577002.0   
Totale immobilizzazioni materiali                      342717358.0   
Totale immobilizzazioni finanziarie                    103068871.0   
Totale patrimonio netto                                305929558.0   
Totale disponibilità liquide a inizio esercizio        106847278.0   
Totale debiti verso banche                             191590518.0   
Differenza tra valore e costi della produzione          22431477.0   
Totale ammortamenti e svalutazioni                     -31210733.0   
Flusso finanziario dell'attività operativa              50640501.0   
Totale flusso finanziario dell'attività di inve...     -60736793.0   
Flusso finanziario dell'attività di finanziamento              NaN   

                                                    30 giugno 2024  
Totale immobilizzazioni immateriali                     47325342.0  
Totale immobilizzazio

/tmp/ipykernel_8143/3462529323.py:20: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  return df.applymap(to_numeric_it)
